In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from unsampled_truth.metrics.anchors import anchor_distribution
from unsampled_truth.metrics.distribution_measures import generalized_jensen_shannon_divergence, average_pairwise_wasserstein_distance
import re

In [ ]:
# Define the ordinal mapping required for the 1-Wasserstein distance
ordinal_mapping = {
    "Strongly Disagree": 1.0,
    "Disagree": 2.0,
    "Neutral": 3.0,
    "Agree": 4.0,
    "Strongly Agree": 5.0
}

In [ ]:
def load_trials(filepath=""):
    trials = []
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line)
            # reduce the size by rounding probabilities to 4 decimal places
            data["probabilities"] = {k: round(v, 4) for k, v in data["probabilities"].items()}
            trials.append(data)
    return trials

In [ ]:
def evaluate_all_models(base_dir: str, questionnaires: list = ["BFI", "SD3"], baseline_personas:list=["antonia", "donovan", "kyle", "nicole", "renate"], num_items: int = 44):
    """
    Dynamically discovers model outputs, lazy-loads them to preserve memory, 
    and applies the exact filtering and metric calculations specified.
    """
    results = []
    base_path = Path(base_dir)
    
    # 1. Dynamic Discovery
    for filepath in base_path.rglob("*_raw.jsonl"):
        
        # Extract metadata directly from the filename
        parts = filepath.stem.split('_')
        if len(parts) < 2:
            continue
            
        baseline_persona = parts[0].capitalize()
        model_name = parts[1]                 

        if baseline_persona.lower() not in [p.lower() for p in baseline_personas]:
            continue
        
        # 2. Lazy Loading
        # Load only one file into memory at a time
        trials = load_trials(filepath)
        
        # 3. Your Exact Filtering and Metric Logic
        for questionnaire in questionnaires:
            for i in range(num_items):
                # Using your exact string matching logic
                item_trials = [trial for trial in trials if trial.get("item_id", "").startswith(f"{questionnaire}{i}_")]
                
                if not item_trials:
                    continue

                distributions = [anchor_distribution([t]) for t in item_trials]
                # kick empty distributions out
                distributions = [d for d in distributions if sum(d.values()) > 0]
                apwd_score = average_pairwise_wasserstein_distance(distributions, ordinal_mapping)
                oom = sum([t["off_option_mass"] for t in item_trials]) / len(item_trials)
                results.append({
                    "model": model_name,       # <--- New tracking variable for downstream comparison
                    "item_id": item_trials[0].get("item_id"),
                    "apwd_score": apwd_score,
                    "baseline_persona": baseline_persona,   # <--- Dynamically assigned
                    "questionnaire": questionnaire,
                    "oom": oom
                })

    return pd.DataFrame(results)

In [ ]:
def extract_model_info(model_name: str) -> tuple[str, str]:
    """
    Extracts the model family and parameter size from heterogeneous naming schemas.
    """
    # Matches strings like '8B', '1.7B', or 'E4B'
    size_match = re.search(r'([a-zA-Z]?\d+(?:\.\d+)?B)', model_name, re.IGNORECASE)
    size = size_match.group(1).upper() if size_match else "Unknown"

    size = float(size.replace('B', '').replace('E', ''))  # Convert to numeric value for sorting
    
    # Extracts the leading identifier as the model family
    family = model_name.split('-')[0]
    
    return family, size

In [ ]:
df = evaluate_all_models("../../outputs/raw_results/")

In [ ]:
# Get apwd scores for each model including standard deviation
apwd_scores = df.groupby(["model", "questionnaire"], as_index=False).agg(
    apwd_mean=("apwd_score", "mean"),
    apwd_std=("apwd_score", "std"),
    oom_mean=("oom", "mean"),
    oom_std=("oom", "std")
)
apwd_scores[["model_family", "parameter_size"]] = apwd_scores["model"].apply(lambda x: pd.Series(extract_model_info(x)))
apwd_scores = apwd_scores.sort_values(by="parameter_size", ascending=True)
print(apwd_scores)

In [ ]:
# Create formatted strings combining Mean \pm Std
apwd_scores["apwd_formatted"] = apwd_scores.apply(
    lambda r: f"{r['apwd_mean']:.3f} ($\pm$ {r['apwd_std']:.3f})", axis=1
)
apwd_scores["oom_formatted"] = apwd_scores.apply(
    lambda r: f"{r['oom_mean']:.3f} ($\pm$ {r['oom_std']:.3f})", axis=1
)

# 1. Generate APWD-only table (Mean ± SD)
apwd_pivot = apwd_scores.pivot(
    index=["model_family", "parameter_size", "model"], 
    columns="questionnaire", 
    values="apwd_formatted"
).reset_index()

apwd_pivot = apwd_pivot.sort_values(by=["model_family", "parameter_size"])
apwd_table = apwd_pivot.drop(columns=["model_family", "parameter_size"])

latex_apwd = apwd_table.to_latex(
    index=False,
    escape=False,
    column_format="l" + "c" * (len(apwd_table.columns) - 1),
    caption="APWD Scores per Questionnaire Across Evaluated Models",
    label="tab:apwd_per_questionnaire"
)

# 2. Generate APWD and OOM table
# Combine the two formatted strings into one cell for a compact multi-metric table
apwd_scores["combined_metrics"] = (
    apwd_scores["apwd_formatted"] + " / " + apwd_scores["oom_formatted"]
)

combined_pivot = apwd_scores.pivot(
    index=["model_family", "parameter_size", "model"], 
    columns="questionnaire", 
    values="combined_metrics"
).reset_index()

combined_pivot = combined_pivot.sort_values(by=["model_family", "parameter_size"])
combined_table = combined_pivot.drop(columns=["model_family", "parameter_size"])

latex_combined = combined_table.to_latex(
    index=False,
    escape=False,
    column_format="l" + "c" * (len(combined_table.columns) - 1),
    caption="APWD and OOM Rates (APWD / OOM) per Questionnaire Across Evaluated Models",
    label="tab:apwd_oom_per_questionnaire"
)

print(latex_apwd)
print(latex_combined)